In [5]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
import joblib

# Load the CSV data
data = pd.read_csv("NEW_URL_DATASET.csv")

# Normalize labels to lowercase and strip whitespace
data['Label'] = data['Label'].astype(str).str.strip().str.lower()

# Build a canonical URL per label
def most_frequent_or_first(s: pd.Series) -> str:
    try:
        m = s.mode()
        if len(m) > 0:
            return m.iloc[0]
    except Exception:
        pass
    return s.iloc[0]

label_to_url = data.groupby('Label')['URL'].apply(most_frequent_or_first).to_dict()

# Features: label text; Targets: canonical URL for that label
X = data['Label']
y = data['Label'].map(label_to_url)

# Split ensuring each label distribution is represented in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Character-level ngrams work well for short labels
vectorizer = CountVectorizer(analyzer='char', ngram_range=(1, 3))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train a simple Naive Bayes classifier
classifier = MultinomialNB()
classifier.fit(X_train_vec, y_train)

# Persist artifacts locally
joblib.dump(classifier, "predict_url.joblib")
joblib.dump(vectorizer, "vectorizer.joblib")

# Also copy updated artifacts to FlaskBackend
try:
    backend_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", "FlaskBackend"))
    shutil.copy2("predict_url.joblib", os.path.join(backend_dir, "predict_url.joblib"))
    shutil.copy2("vectorizer.joblib", os.path.join(backend_dir, "vectorizer.joblib"))
    print(f"Copied artifacts to: {backend_dir}")
except Exception as e:
    print(f"Copy to FlaskBackend skipped/failed: {e}")

# Evaluate the classifier
accuracy = accuracy_score(y_test, classifier.predict(X_test_vec))
print("Accuracy:", accuracy)


Copied artifacts to: E:\SignLens-Project\FlaskBackend
Accuracy: 0.8432432432432433


In [6]:
# Load the saved model and vectorizer
loaded_classifier = joblib.load("predict_url.joblib")
loaded_vectorizer = joblib.load("vectorizer.joblib")

# Make predictions using the loaded model
for sample in ["a", "ae", "capital_l", "amma", "chocolate"]:
    predicted_url_new = loaded_classifier.predict(loaded_vectorizer.transform([sample]))
    print(sample, "->", predicted_url_new[0])


a -> https://drive.google.com/file/d/1-TD3yCD9EdajeADS3rKBdRvpY_waGYe3/edit
ae -> https://drive.google.com/file/d/170WZCK_kN9F6No6TvDMZKV3Mb7ylgyeh/edit
capital_l -> https://drive.google.com/file/d/15Grpht-5_qE0ekP2DbsW-mH-QRbykoOw/edit
amma -> https://drive.google.com/file/d/12ujDbZDJyL4IKJ1oeHNoU-oSa8CCQt6z/edit
chocolate -> https://drive.google.com/file/d/11aA8DPPsHUE_uGzjjfCci4eV3wfP3aRN/edit


In [7]:
import joblib

# Load artifacts explicitly and test a single input
loaded_classifier = joblib.load("predict_url.joblib")
loaded_vectorizer = joblib.load("vectorizer.joblib")

new_label_to_predict = "ae"
predicted_url_new = loaded_classifier.predict(loaded_vectorizer.transform([new_label_to_predict]))
print("Predicted URL for new label:", predicted_url_new[0])


Predicted URL for new label: https://drive.google.com/file/d/170WZCK_kN9F6No6TvDMZKV3Mb7ylgyeh/edit


In [8]:
#Inspect predictions on the held-out test set
from sklearn.metrics import classification_report

y_pred = classifier.predict(X_test_vec)
print(classification_report(y_test, y_pred, zero_division=0))


                                                                        precision    recall  f1-score   support

https://drive.google.com/file/d/1--Ugp5sKF8NCfKmfwNKNVsjcCul4VwmT/edit       1.00      1.00      1.00         4
https://drive.google.com/file/d/1-1r-IMYPfpvSozBaEut97n_MfLD2js9t/edit       1.00      1.00      1.00         2
https://drive.google.com/file/d/1-TD3yCD9EdajeADS3rKBdRvpY_waGYe3/edit       0.33      1.00      0.50         4
https://drive.google.com/file/d/1-VeJs1xGJ50QDSqv2eaUnOO-A1Rtp-5D/edit       1.00      1.00      1.00         4
https://drive.google.com/file/d/1-YQR1-8m8iIu2-Ns69mTYh2T6nroNkyd/edit       1.00      1.00      1.00         4
https://drive.google.com/file/d/1-_uXs7jkY49D27l_lzkGjd22ok6f_ur5/edit       1.00      1.00      1.00         1
https://drive.google.com/file/d/1-eBsiUjIDwewcVCHqpKj5yTwTRg3yMWX/edit       1.00      1.00      1.00         2
https://drive.google.com/file/d/1-mQsNjpO_2G5ZIOUffdK-WuoXjMWjxpX/edit       0.50      1.00      0.67  